# SHAWAF Stage 1 — TVPReid text-to-video eval

Frozen VLM encoders (X-CLIP, LanguageBind, InternVideo2 CLIP-S) on the unofficial Hub mirror **[bassatbassat/TVPReid](https://huggingface.co/datasets/bassatbassat/TVPReid)**.

This notebook **imports** `shawaf_vlm`; it does not reimplement the eval loop.

**Kaggle:** enable **GPU** + **Internet**. After any CUDA crash, **Restart session** then **Run All from the top.** The install cell always `git fetch` + `reset --hard origin/main` into `/kaggle/working/shawaf-vlm` and must print `shawaf_vlm 0.1.15`. If it prints an older version, re-run that cell.

Benchmark of **3 models × sampling families × pooling**, with CLIP4Clip / TVPR / Re-ID metrics plus runtime:

- `uniform8` — 8 frames across the tracklet (old Stage 1)
- `vt_1fps_n12` — [CLIP4Clip](https://arxiv.org/abs/2104.08860) sampling (1 fps, 12-frame cap)
- `vt_2fps_n32` — same idea, denser coverage
- `reid_8fps_n64` — [TVPR](https://arxiv.org/abs/2307.07184)-style consecutive clips (8 fps, 64-frame cap)

Reported: Rank-1/5/10/20/50, mAP, MdR, MnR, nDCG@10, mINP, decode/video/text/score seconds, ms/item, peak GPU GB. Each window config encodes 8-frame clips at stride 4 **once**, then scores `mean` / `mean_s8` / `max` / `query_max`. Comment out a dict in `WINDOW_CONFIGS` to skip it.

The notebook clones [this GitHub repo](https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-) and downloads only the **test** mp4s from **[bassatbassat/TVPReid](https://huggingface.co/datasets/bassatbassat/TVPReid)** (not train/val). Cite Zhang et al. (ACM MM 2024) plus the PRID / iLIDS / Duke source papers. Drop `"duke"` from `SUBSETS` for a much smaller download.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
REPO_DIR = "/kaggle/working/shawaf-vlm"
PACKAGE_DIR = ""  # optional local / Kaggle dataset path that already contains shawaf_vlm

HF_REPO = "bassatbassat/TVPReid"
SPLIT = "test"
# Test-only download. Drop "duke" (~600 videos) for a faster smoke run.
SUBSETS = ["prid", "ilids", "duke"]

MODELS = ["xclip", "languagebind", "internvideo2"]
RUN_CLIP_1B = False
if RUN_CLIP_1B:
    MODELS.append("internvideo2_clip_1b")

NUM_FRAMES = 8
RUN_UNIFORM = True
# Comment out a config to skip it. Each extra fps/max_frames pair is another encode.
WINDOW_CONFIGS = [
    {
        "name": "vt_1fps_n12",  # CLIP4Clip paper default
        "sample_fps": 1.0,
        "max_frames": 12,
        "stride": 4,
        "pools": ("mean", "mean_s8", "max", "query_max"),
    },
    {
        "name": "vt_2fps_n32",  # denser sparse coverage
        "sample_fps": 2.0,
        "max_frames": 32,
        "stride": 4,
        "pools": ("mean", "mean_s8", "max", "query_max"),
    },
    {
        "name": "reid_8fps_n64",  # TVPR-style consecutive clips
        "sample_fps": 8.0,
        "max_frames": 64,
        "stride": 4,
        "pools": ("mean", "mean_s8", "max", "query_max"),
    },
]
BATCH_SIZE = 4
TEXT_BATCH_SIZE = 32
DEVICE = "cuda"

_kaggle_work = Path("/kaggle/working")
_work = _kaggle_work if _kaggle_work.is_dir() else Path.cwd()
FRAME_CACHE = _work / "tvpreid_frames"
RESULTS_DIR = _work / "results"


In [ ]:
import os
import shutil
import sys
import subprocess
from pathlib import Path


def _has_package(root: Path) -> bool:
    return (root / "shawaf_vlm").is_dir() and (root / "pyproject.toml").is_file()


def find_repo_root() -> Path:
    kaggle = Path("/kaggle/working").is_dir()
    repo = Path(REPO_DIR)

    # On Kaggle always refresh /kaggle/working/shawaf-vlm. cwd already contains
    # shawaf_vlm after the first run, so a local-first search would skip git pull.
    if kaggle:
        if (repo / ".git").is_dir():
            subprocess.check_call(
                ["git", "-C", str(repo), "fetch", "--depth", "1", "origin", "main"]
            )
            subprocess.check_call(
                ["git", "-C", str(repo), "reset", "--hard", "origin/main"]
            )
            return repo
        if repo.exists():
            shutil.rmtree(repo)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo)])
        return repo

    if PACKAGE_DIR and _has_package(Path(PACKAGE_DIR)):
        return Path(PACKAGE_DIR)
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if _has_package(candidate):
            return candidate
    if _has_package(repo):
        return repo
    raise RuntimeError(f"shawaf_vlm not found. Clone {REPO_URL}")


repo = find_repo_root()
os.chdir(repo)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[all]"])

# Drop stale imports from an earlier notebook run, including a spec-less
# flash_attn stub left behind by a failed InternVideo2 load.
for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]
    if name == "flash_attn" or name.startswith("flash_attn."):
        del sys.modules[name]
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import shawaf_vlm

print("package root", repo, flush=True)
print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)

In [ ]:
from shawaf_vlm.data.tvpreid import (
    discover_local_tvpreid,
    download_tvpreid,
    load_tvpreid_from_root,
)

local_root = discover_local_tvpreid()
if local_root is None:
    print("Downloading", HF_REPO, "split", SPLIT, "subsets", SUBSETS, flush=True)
    tvpreid_root = download_tvpreid(
        repo_id=HF_REPO,
        configs=tuple(SUBSETS),
        split=SPLIT,
        max_workers=8,
    )
else:
    tvpreid_root = local_root
    print("Using local TVPReid at", tvpreid_root, flush=True)

FRAME_CACHE.mkdir(parents=True, exist_ok=True)
splits_by_subset = {}
for subset in SUBSETS:
    splits = load_tvpreid_from_root(tvpreid_root, config=subset, split=SPLIT)
    splits_by_subset[subset] = splits
    print(
        f"{subset:6}  queries={len(splits.query):4}  "
        f"gallery={len(splits.gallery):4}  source={splits.source}"
    )

In [ ]:
import json
from pathlib import Path

import torch

from shawaf_vlm.eval_loop import (
    evaluate_text_to_tracklet,
    evaluate_text_to_tracklet_windows,
)
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models import all_specs, build_encoder
from shawaf_vlm.models.runtime import ensure_cuda_healthy

print("torch", torch.__version__, "cuda", torch.cuda.is_available(), flush=True)
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is off. Turn on GPU and rerun from the top.")
if torch.cuda.is_available():
    print("GPU", torch.cuda.get_device_name(0), flush=True)
ensure_cuda_healthy(DEVICE)

results = []
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def _record(spec, subset, splits, protocol, metrics, *, stride=None, sample_fps=None, max_frames=None):
    payload = {
        "model_key": spec.key,
        "model": spec.label,
        "checkpoint": spec.checkpoint,
        "dataset": "tvpreid",
        "hub_repo": HF_REPO,
        "subset": subset,
        "split": SPLIT,
        "split_source": splits.source,
        "protocol": protocol,
        "num_queries": len(splits.query),
        "num_gallery": len(splits.gallery),
        "num_frames": NUM_FRAMES,
        "stride": stride,
        "sample_fps": sample_fps,
        "max_frames": max_frames,
        "metrics": metrics,
    }
    path = RESULTS_DIR / f"{spec.key}_tvpreid_{subset}_{protocol.replace('/', '-')}.json"
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    results.append(payload)


for name in MODELS:
    spec = all_specs()[name]
    print("\n===", spec.label, "===")
    encoder = build_encoder(name, device=DEVICE)
    for subset, splits in splits_by_subset.items():
        print(f"\n--- {subset} ---")
        if RUN_UNIFORM:
            metrics = evaluate_text_to_tracklet(
                encoder=encoder,
                splits=splits,
                num_frames=NUM_FRAMES,
                batch_size=BATCH_SIZE,
                text_batch_size=TEXT_BATCH_SIZE,
                frame_cache=FRAME_CACHE,
            )
            print("[uniform8]")
            print(format_metrics(metrics))
            _record(spec, subset, splits, "uniform8", metrics)
        for cfg in WINDOW_CONFIGS:
            label = cfg["name"]
            print(
                f"[{label}] fps={cfg['sample_fps']:g} max={cfg['max_frames']} "
                f"stride={cfg['stride']}",
                flush=True,
            )
            scored = evaluate_text_to_tracklet_windows(
                encoder=encoder,
                splits=splits,
                num_frames=NUM_FRAMES,
                stride=cfg["stride"],
                sample_fps=cfg["sample_fps"],
                max_frames=cfg["max_frames"],
                pools=cfg["pools"],
                batch_size=BATCH_SIZE,
                text_batch_size=TEXT_BATCH_SIZE,
                frame_cache=FRAME_CACHE,
            )
            for pool, metrics in scored.items():
                protocol = f"{label}/{pool}"
                print(f"[{protocol}]")
                print(format_metrics(metrics))
                _record(
                    spec,
                    subset,
                    splits,
                    protocol,
                    metrics,
                    stride=cfg["stride"],
                    sample_fps=cfg["sample_fps"],
                    max_frames=cfg["max_frames"],
                )
    del encoder


In [ ]:
from collections import defaultdict

print(
    f"{'model':<16} {'subset':<8} {'protocol':<28} "
    f"{'R1':>6} {'R5':>6} {'R10':>6} {'R20':>6} {'R50':>6} "
    f"{'mAP':>6} {'MdR':>5} {'MnR':>6} {'nDCG':>6} {'mINP':>6} "
    f"{'vid_s':>7} {'tot_s':>7} {'ms/it':>7} {'GPU':>6} {'Q':>5}"
)
for row in results:
    m = row["metrics"]
    print(
        f"{row['model_key']:<16} {row['subset']:<8} {row['protocol']:<28} "
        f"{m['Rank-1']:6.2f} {m['Rank-5']:6.2f} {m['Rank-10']:6.2f} "
        f"{m.get('Rank-20', 0):6.2f} {m.get('Rank-50', 0):6.2f} "
        f"{m['mAP']:6.2f} {m.get('MdR', 0):5.1f} {m.get('MnR', 0):6.1f} "
        f"{m.get('nDCG@10', 0):6.2f} {m.get('mINP', 0):6.2f} "
        f"{m.get('video_s', 0):7.1f} {m.get('total_s', 0):7.1f} "
        f"{m.get('video_ms_per_item', 0):7.1f} {m.get('peak_gpu_gb', 0):6.2f} "
        f"{int(m['num_valid_queries']):5d}"
    )

protocols = []
for row in results:
    if row["protocol"] not in protocols:
        protocols.append(row["protocol"])

grouped = defaultdict(dict)
for row in results:
    grouped[(row["model_key"], row["subset"])][row["protocol"]] = row["metrics"]


def _print_grid(title, key, fmt=".2f"):
    print(f"\n{title}")
    print(f"{'model':<16} {'subset':<8}", end="")
    for protocol in protocols:
        print(f" {protocol[-16:]:>16}", end="")
    print()
    for group_key in grouped:
        model_key, subset = group_key
        print(f"{model_key:<16} {subset:<8}", end="")
        for protocol in protocols:
            metrics = grouped[group_key].get(protocol)
            value = format(metrics[key], fmt) if metrics and key in metrics else "-"
            print(f" {value:>16}", end="")
        print()


_print_grid("Rank-1 by protocol", "Rank-1")
_print_grid("mAP by protocol", "mAP")
_print_grid("MdR by protocol", "MdR", ".1f")
_print_grid("video seconds by protocol", "video_s", ".1f")
_print_grid("peak GPU GB by protocol", "peak_gpu_gb")
